Step 1: Bootstrap and install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
from google.colab import userdata, drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')

Step 2: Install & verify LangGraph

In [ ]:
!pip install --quiet langgraph

import importlib.metadata
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Check version
try:
    print(f"✓ langgraph {importlib.metadata.version('langgraph')}")
except importlib.metadata.PackageNotFoundError:
    raise RuntimeError("langgraph not installed — run !pip install langgraph")

# 2. Build and run a minimal graph to verify imports and execution
class _State(TypedDict):
    status: str

def _node(state: _State) -> _State:
    return {"status": "ok"}

_g = StateGraph(_State)
_g.add_node("test", _node)
_g.add_edge(START, "test")
_g.add_edge("test", END)

result = _g.compile().invoke({"status": "start"})

if result.get("status") != "ok":
    raise AssertionError(f"Expected status 'ok', got {result}")

print("✓ Graph compiled and executed successfully!")

Step 3: Pipeline

In [ ]:
from astra_swarm.graph import graph_triage, triage_graph
from astra_swarm.correlation import correlate
from astra_swarm.cassette import cassette
from IPython.display import Image, display
import json
from pathlib import Path

# Show the compiled graph
display(Image(triage_graph.get_graph().draw_mermaid_png()))

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/02_alerts.json").read_text()
)

# Run the full graph
with cassette("week3_milestone"):
    triaged = [graph_triage(a) for a in alerts]

# Correlate into incidents
incidents = correlate(triaged)

# Persist
out = Path("/content/astra-swarm/data/synthetic/week03_triage_results.json")
out.write_text(json.dumps([
    {**{k: v.model_dump() if hasattr(v, 'model_dump') else v for k, v in t.items()}}
    for t in triaged
], indent=2, default=str))

inc_out = Path("/content/astra-swarm/data/synthetic/week03_incidents.json")
inc_out.write_text(json.dumps([i.model_dump() for i in incidents], indent=2, default=str))

Step 4: Metrics

In [ ]:
from collections import Counter

# Refinement rate — how often did the evaluator send back for revision?
refined = sum(1 for t in triaged if t.get("refinement_count", 0) > 0)
print(f"Refinement rate: {refined}/{len(triaged)} ({refined/len(triaged):.0%})")

# Worker fan-out — which workers actually ran per alert
worker_dist = Counter(tuple(sorted(t.get("workers_run", []))) for t in triaged)
print(f"Worker combinations: {dict(worker_dist)}")

# Evaluator scores
scores = [t["evaluation"] for t in triaged if "evaluation" in t]
avg_completeness = sum(s.completeness for s in scores) / len(scores)
avg_citation = sum(s.citation_quality for s in scores) / len(scores)
avg_severity = sum(s.severity_defensibility for s in scores) / len(scores)
print(f"Avg eval scores: completeness={avg_completeness:.2f}, "
      f"citation={avg_citation:.2f}, severity_def={avg_severity:.2f}")

# Correlation collapse
print(f"Correlation: {len(alerts)} alerts → {len(incidents)} incidents")
multi = [i for i in incidents if i.alert_count > 1]
print(f"  Multi-alert incidents: {len(multi)}")